# Ablation Experiments

This notebook runs targeted ablation experiments to investigate:
1. **Continuation ablation** — disable `continuation=True` so all methods reset weights per lambda stage
2. **Single lambda** — run only lambda=0.5 to avoid cross-stage effects
3. **Alternative fairness measure** — compare MAD vs Gap vs Atkinson
4. **Gradient cosine table** — print per-iteration gradient diagnostics

### Data split: 50% train / 50% test (no validation)

### Training note: Full-batch SGD
All methods use **full-batch gradient descent** (`batch_size=-1` in `DEFAULT_TRAIN_CFG`).
This means every gradient step uses the entire training set — there is no mini-batch stochasticity.
The optimizer is SGD (default) or Adam, applied identically to all methods.
The only difference between methods is **which gradients are combined** and **how**.

## Experiment Design Notes

### Why FDFL starts with positive cos(g_dec, g_fair) while other methods start negative

In the gradient cosine similarity plots at lambda=0.2, FDFL shows a positive
cos(g_dec, g_fair) at iteration 0 (~0.8), while WS-equal, WS-dec, MGDA, PCGrad,
and FFO all start negative (~-0.3). This is **not a bug** — it is a direct
consequence of the **continuation** setting.

Most methods (WS-equal, WS-dec, MGDA, PCGrad, FFO) use `continuation=True`:
model weights carry over across lambda stages (0.0 -> 0.5, with 70 training steps each). FDFL is the exception with `continuation=False`, meaning
it resets to random initialization at every lambda stage.

When we plot at lambda=0.5 (the 2nd stage):
- **Continuation methods** enter this stage having already trained for 70 steps
  (at lambda=0.0). The model is near the Pareto frontier
  where improving decisions further comes at the cost of group equity — hence
  g_dec and g_fair **conflict** (negative cosine).
- **FDFL** starts lambda=0.2 from fresh random initialization. At random weights,
  both "improve decisions" and "equalize across groups" push predictions in similar
  directions away from noise — hence they **align** (positive cosine). The conflict
  only emerges after several training steps.

This notebook's Experiment 1 confirms this: when we disable continuation for all
methods, they all start with similar positive cosine at each lambda stage.

### The continuation computational shortcut

Continuation (`continuation=True`) serves two purposes:
1. **Warm-starting**: the model does not waste steps re-learning basic prediction
   structure at higher lambda values.
2. **Pareto frontier tracing**: by gradually increasing the fairness penalty lambda,
   we trace out points along the Pareto frontier. Each stage refines the previous
   solution rather than solving from scratch.

The trade-off is path-dependence: the lambda=0.5 result depends on the full
training history (0.0 -> 0.5). Experiments 1 and 2 in this
notebook isolate methods from this effect.

### Other implementation details

- **FPTO warmstart** (`warmstart_fraction=0.25`): PLG and FPLG run the first 25%
  of steps as pure FPTO (prediction + fairness only, no decision gradient). This
  stabilizes the predictor before introducing the noisier decision gradient.
- **Alpha schedule** (inv_sqrt decay, alpha0=1.0): the prediction gradient weight
  decays as alpha(t) = 1/sqrt(t+1). Early training is prediction-dominated; later
  steps focus on decision quality.
- **Full-batch training** (`batch_size=-1`): every step uses the entire training set.
  Required because the resource allocation solver needs all patients simultaneously
  to respect the global budget constraint. Also eliminates mini-batch noise from
  gradient cosine measurements.
- **Gradient clipping** (`grad_clip_norm=10000`): safety net against exploding
  gradients, set high enough that it rarely activates.
- **Fairness smoothing** (`fairness_smoothing=1e-6`): epsilon in fairness metric
  denominators to avoid division by zero.
- **force_lambda_path_all_methods=True**: even methods without a fairness objective
  run through all lambda stages, ensuring every method produces results at every
  lambda for fair comparison.

In [ ]:
# Cell 1: Install dependencies (Colab only)
!pip install -q torch numpy pandas scipy matplotlib cvxpy mosek
!pip install -q git+https://github.com/cvxpy/cvxtorch.git
!pip install -q --upgrade threadpoolctl

In [ ]:
# Cell 2: Path setup
import os, sys, copy, importlib
import numpy as np
import pandas as pd

# --- Colab: mount drive ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/colab_upload'

# Verify path exists
if not os.path.exists(DRIVE_ROOT):
    print(f'ERROR: {DRIVE_ROOT} not found. Please check the path.')

MOSEK_LIC_PATH = os.path.join(DRIVE_ROOT, 'data', 'mosek.lic')
if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')

# Add DRIVE_ROOT and src to sys.path
for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Force reload all fair_dfl modules to pick up any Drive file changes
mods_to_remove = [k for k in sys.modules if k.startswith('fair_dfl') or k in ('configs', 'plotting', 'analysis')]
for k in mods_to_remove:
    del sys.modules[k]

# CRITICAL: Change to DRIVE_ROOT so local imports work
os.chdir(DRIVE_ROOT)
print(f'Working directory changed to: {os.getcwd()}')

In [ ]:
# Cell 3: Imports and helpers
from configs import ALL_METHOD_CONFIGS, DEFAULT_TRAIN_CFG, make_task_cfg, describe_method
from fair_dfl.runner import run_experiment_unified
from fair_dfl.training.method_spec import resolve_method_spec
from plotting import plot_all, plot_gradient_conflict, plot_training_curves
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.join(DRIVE_ROOT, 'results', 'ablation')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Methods to compare in ablation
ABLATION_METHODS = ['FPTO', 'FDFL', 'FFO', 'WS-equal', 'WS-dec', 'MGDA', 'PCGrad']

DATA_CSV = os.path.join(DRIVE_ROOT, 'data', 'data_processed.csv')
ALPHA = 2.0
N_SAMPLE = 1000  # use 0 for full data


def make_summary_table(sdf, group_cols=None):
    """Aggregate test metrics as mean +/- std across seeds.

    Returns a DataFrame with one row per unique combination of group_cols,
    with formatted 'mean +/- std' columns for regret, fairness, MSE.
    """
    if group_cols is None:
        group_cols = ['config_name']
    metrics = ['test_regret', 'test_fairness', 'test_pred_mse']
    short = {'test_regret': 'Regret', 'test_fairness': 'Fairness', 'test_pred_mse': 'Pred MSE'}
    rows = []
    for keys, grp in sdf.groupby(group_cols):
        if isinstance(keys, str):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        for m in metrics:
            if m not in grp.columns:
                continue
            mu, sd = grp[m].mean(), grp[m].std()
            row[f'{short[m]}'] = f'{mu:.4f} +/- {sd:.4f}'
            row[f'{short[m]}_mean'] = mu
        row['n_seeds'] = len(grp)
        rows.append(row)
    return pd.DataFrame(rows)


print(f'Methods: {ABLATION_METHODS}')
print(f'Results dir: {RESULTS_DIR}')

---
## Experiment 1: Disable Continuation

By default, FPLG/WS-equal/WS-dec/MGDA/PCGrad/FFO have `continuation=True`,
meaning model weights carry over across lambda stages (0.0 -> 0.5).
FDFL has `continuation=False` and resets to random init each stage.

This explains why gradient cosines start at different values — continuation methods
enter lambda=0.5 already trained for 70 steps, while FDFL starts fresh.

Here we **disable continuation for all methods** to see if they all start the same.

In [ ]:
# Cell 4: Run with continuation=False for all methods

methods_no_cont = {}
for name in ABLATION_METHODS:
    cfg = copy.deepcopy(ALL_METHOD_CONFIGS[name])
    cfg['continuation'] = False  # force reset at each lambda stage
    methods_no_cont[name] = cfg

print('All methods now have continuation=False:')
for name, cfg in methods_no_cont.items():
    spec = resolve_method_spec(cfg)
    print(f'  {name:15s}  continuation={spec.continuation}')

task_cfg = make_task_cfg(data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA)
train_cfg = copy.deepcopy(DEFAULT_TRAIN_CFG)
train_cfg['log_every'] = 1  # log every iteration for detailed cosine tracking

exp_cfg = {
    'task': task_cfg,
    'training': train_cfg,
}

stage_df_nocont, iter_df_nocont = run_experiment_unified(exp_cfg, method_configs=methods_no_cont)
stage_df_nocont['config_name'] = stage_df_nocont['method'].str.upper()
# Map method names back to original config names for plotting
_method_to_config = {name.lower(): name for name in ABLATION_METHODS}
stage_df_nocont['config_name'] = stage_df_nocont['method'].map(
    lambda x: _method_to_config.get(x, x.upper()))
iter_df_nocont['config_name'] = iter_df_nocont['method'].map(
    lambda x: _method_to_config.get(x, x.upper()))
stage_df_nocont['alpha_fair'] = ALPHA
iter_df_nocont['alpha_fair'] = ALPHA

print(f'\nDone. {len(stage_df_nocont)} stage rows, {len(iter_df_nocont)} iter rows.')

In [ ]:
# Cell 5: Plot gradient cosines — all methods should now start at same point
nocont_dir = os.path.join(RESULTS_DIR, 'no_continuation')
os.makedirs(nocont_dir, exist_ok=True)

plot_gradient_conflict(iter_df_nocont, alpha_values=[ALPHA], results_dir=nocont_dir)
plot_training_curves(iter_df_nocont, alpha_values=[ALPHA], results_dir=nocont_dir)

In [ ]:
# Cell 5b: Exp 1 summary — per method, per lambda (mean +/- std across seeds)

print(f'=== Experiment 1: No Continuation (alpha={ALPHA}) ===')
print(f'Each cell shows mean +/- std across {len(DEFAULT_TRAIN_CFG["seeds"])} seeds\n')

# One table per lambda value
for lam in sorted(stage_df_nocont['lambda'].unique()):
    sub = stage_df_nocont[stage_df_nocont['lambda'] == lam]
    tbl = make_summary_table(sub)
    tbl = tbl.sort_values('Regret_mean')

    print(f'--- lambda = {lam} ---')
    display_cols = ['config_name', 'Regret', 'Fairness', 'Pred MSE']
    print(tbl[[c for c in display_cols if c in tbl.columns]].to_string(index=False))
    print()

# Best lambda selection per method
print('--- Best lambda per method (minimizes normalized regret + fairness) ---')
best_rows = []
for method in sorted(stage_df_nocont['config_name'].unique()):
    msub = stage_df_nocont[stage_df_nocont['config_name'] == method]
    per_lam = msub.groupby('lambda').agg(
        regret=('test_regret', 'mean'),
        fairness=('test_fairness', 'mean'),
        mse=('test_pred_mse', 'mean'),
    ).reset_index()
    if len(per_lam) == 0:
        continue
    r_max = max(per_lam['regret'].max(), 1e-12)
    f_max = max(per_lam['fairness'].max(), 1e-12)
    per_lam['score'] = per_lam['regret'] / r_max + per_lam['fairness'] / f_max
    best = per_lam.loc[per_lam['score'].idxmin()]
    best_rows.append({
        'method': method,
        'best_lambda': best['lambda'],
        'regret': f'{best["regret"]:.4f}',
        'fairness': f'{best["fairness"]:.4f}',
        'mse': f'{best["mse"]:.4f}',
    })

print(pd.DataFrame(best_rows).to_string(index=False))

---
## Experiment 2: Single Lambda = 0.5

Run only lambda=0.5 to eliminate cross-stage effects entirely.
Every method starts from the same random init and runs 70 steps.

In [ ]:
# Cell 6: Run with single lambda=0.5

methods_single_lambda = copy.deepcopy(methods_no_cont)  # also no continuation

train_cfg_single = copy.deepcopy(DEFAULT_TRAIN_CFG)
train_cfg_single['lambdas'] = [0.5]  # single lambda
train_cfg_single['log_every'] = 1

exp_cfg_single = {
    'task': make_task_cfg(data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA),
    'training': train_cfg_single,
}

stage_df_single, iter_df_single = run_experiment_unified(exp_cfg_single, method_configs=methods_single_lambda)
stage_df_single['config_name'] = stage_df_single['method'].map(
    lambda x: _method_to_config.get(x, x.upper()))
iter_df_single['config_name'] = iter_df_single['method'].map(
    lambda x: _method_to_config.get(x, x.upper()))
stage_df_single['alpha_fair'] = ALPHA
iter_df_single['alpha_fair'] = ALPHA

print(f'Done. {len(stage_df_single)} stage rows, {len(iter_df_single)} iter rows.')

In [ ]:
# Cell 7: Plot single-lambda results
single_dir = os.path.join(RESULTS_DIR, 'single_lambda')
os.makedirs(single_dir, exist_ok=True)

plot_gradient_conflict(iter_df_single, alpha_values=[ALPHA], results_dir=single_dir)
plot_training_curves(iter_df_single, alpha_values=[ALPHA], results_dir=single_dir)

In [ ]:
# Cell 7b: Exp 2 summary — single lambda=0.5 (mean +/- std across seeds)

print(f'=== Experiment 2: Single Lambda=0.5 (alpha={ALPHA}) ===')
print(f'Each cell shows mean +/- std across {len(DEFAULT_TRAIN_CFG["seeds"])} seeds\n')

tbl = make_summary_table(stage_df_single)
tbl = tbl.sort_values('Regret_mean')
tbl.insert(0, 'Rank', range(1, len(tbl) + 1))

display_cols = ['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE', 'n_seeds']
print(tbl[[c for c in display_cols if c in tbl.columns]].to_string(index=False))

---
## Experiment 3: Alternative Fairness Measures

Compare the three available prediction-side fairness metrics:
- **MAD** (Mean Absolute Deviation) — default, works for K>=2 groups
- **Gap** (Group Accuracy Parity) — binary group MSE gap
- **Atkinson** (Atkinson Inequality Index) — penalizes extreme disparities

We run the same methods under each fairness metric to see if performance ranking is consistent.

In [ ]:
# Cell 8: Run all three fairness types

fairness_types = ['mad', 'gap', 'atkinson']
results_by_fairness = {}

for ft in fairness_types:
    print(f'\n{"="*60}')
    print(f'Running fairness_type = {ft}')
    print(f'{"="*60}')

    ft_task_cfg = make_task_cfg(
        data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA, fairness_type=ft)
    ft_train_cfg = copy.deepcopy(DEFAULT_TRAIN_CFG)
    ft_train_cfg['lambdas'] = [0.5]
    ft_train_cfg['log_every'] = 1

    ft_exp_cfg = {'task': ft_task_cfg, 'training': ft_train_cfg}
    ft_methods = copy.deepcopy(methods_no_cont)

    stage_df_ft, iter_df_ft = run_experiment_unified(ft_exp_cfg, method_configs=ft_methods)
    stage_df_ft['config_name'] = stage_df_ft['method'].map(
        lambda x: _method_to_config.get(x, x.upper()))
    iter_df_ft['config_name'] = iter_df_ft['method'].map(
        lambda x: _method_to_config.get(x, x.upper()))
    stage_df_ft['alpha_fair'] = ALPHA
    iter_df_ft['alpha_fair'] = ALPHA
    stage_df_ft['fairness_type'] = ft
    iter_df_ft['fairness_type'] = ft

    results_by_fairness[ft] = (stage_df_ft, iter_df_ft)
    print(f'{ft}: {len(stage_df_ft)} stage rows, {len(iter_df_ft)} iter rows')

print('\nAll fairness types complete.')

In [ ]:
# Cell 9: Compare fairness measures — per-type tables + rank consistency

print(f'=== Experiment 3: Fairness Measure Comparison (alpha={ALPHA}, lambda=0.5) ===')
print(f'Each cell shows mean +/- std across {len(DEFAULT_TRAIN_CFG["seeds"])} seeds\n')

# --- One table per fairness type ---
for ft in fairness_types:
    sdf, _ = results_by_fairness[ft]
    tbl = make_summary_table(sdf)
    tbl = tbl.sort_values('Regret_mean')
    tbl.insert(0, 'Rank', range(1, len(tbl) + 1))

    print(f'{"="*80}')
    print(f'  Fairness type: {ft.upper()}')
    print(f'{"="*80}')
    display_cols = ['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE']
    print(tbl[[c for c in display_cols if c in tbl.columns]].to_string(index=False))
    print()

# --- Cross-fairness rank comparison ---
print(f'{"="*80}')
print('  Ranking Consistency Across Fairness Types (by test regret, 1 = best)')
print(f'{"="*80}')

rank_dict = {}
for ft in fairness_types:
    sdf, _ = results_by_fairness[ft]
    tbl = make_summary_table(sdf).sort_values('Regret_mean').reset_index(drop=True)
    for rank, (_, r) in enumerate(tbl.iterrows(), 1):
        method = r['config_name']
        if method not in rank_dict:
            rank_dict[method] = {}
        rank_dict[method][ft.upper()] = rank

rank_table = pd.DataFrame.from_dict(rank_dict, orient='index')
rank_table.index.name = 'method'
rank_table['Avg Rank'] = rank_table.mean(axis=1)
rank_table = rank_table.sort_values('Avg Rank')
print(rank_table.to_string(float_format='%.1f'))

# --- Regret ranking order per fairness type ---
print('\n--- Regret ordering (best -> worst) ---')
for ft in fairness_types:
    sdf, _ = results_by_fairness[ft]
    tbl = make_summary_table(sdf).sort_values('Regret_mean')
    ranking = list(tbl['config_name'].values)
    print(f'  {ft.upper():10s}: {" > ".join(ranking)}')

In [ ]:
# Cell 10: Side-by-side gradient conflict plots for each fairness type
for ft, (sdf, idf) in results_by_fairness.items():
    ft_dir = os.path.join(RESULTS_DIR, f'fairness_{ft}')
    os.makedirs(ft_dir, exist_ok=True)
    print(f'\n--- Gradient conflict: {ft} ---')
    plot_gradient_conflict(idf, alpha_values=[ALPHA], results_dir=ft_dir)
    plot_training_curves(idf, alpha_values=[ALPHA], results_dir=ft_dir)

---
## Gradient Cosine Similarity Table

Print per-iteration gradient cosine values as a table for detailed inspection.
Uses the single-lambda experiment (Experiment 2) data.

In [ ]:
# Cell 11: Gradient cosine table — per method, per iteration

cos_cols = ['cos_dec_pred', 'cos_dec_fair', 'cos_pred_fair']
norm_cols = ['grad_norm_dec', 'grad_norm_pred', 'grad_norm_fair', 'grad_norm_combined']
display_cols = ['iter'] + cos_cols + norm_cols

# Use single-lambda data (iter_df_single)
df = iter_df_single.copy()

# Average across seeds
for method in sorted(df['config_name'].unique()):
    msub = df[df['config_name'] == method]
    available = [c for c in display_cols if c in msub.columns]
    grouped = msub.groupby('iter')[available[1:]].mean().reset_index()
    grouped.insert(0, 'iter', grouped.index if 'iter' not in grouped.columns else grouped['iter'])

    print(f'\n{"="*100}')
    print(f'Method: {method}  |  lambda=0.5  |  alpha_fair={ALPHA}  |  averaged over seeds')
    print(f'{"="*100}')

    # Format for readability
    fmt_df = grouped.copy()
    for c in cos_cols:
        if c in fmt_df.columns:
            fmt_df[c] = fmt_df[c].map(lambda x: f'{x:+.4f}')
    for c in norm_cols:
        if c in fmt_df.columns:
            fmt_df[c] = fmt_df[c].map(lambda x: f'{x:.6f}')

    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
        print(fmt_df[[c for c in display_cols if c in fmt_df.columns]].to_string(index=False))

In [ ]:
# Cell 12: Summary comparison table — iteration 0 vs final iteration

df = iter_df_single.copy()
compare_rows = []

for method in sorted(df['config_name'].unique()):
    msub = df[df['config_name'] == method]
    iters = sorted(msub['iter'].unique())
    if len(iters) < 2:
        continue
    first_iter, last_iter = iters[0], iters[-1]

    for label, it in [('start', first_iter), ('end', last_iter)]:
        isub = msub[msub['iter'] == it]
        row = {'method': method, 'phase': label, 'iter': it}
        for c in cos_cols + norm_cols + ['loss_dec', 'loss_pred', 'loss_fair']:
            if c in isub.columns:
                row[c] = isub[c].mean()
        compare_rows.append(row)

compare_df = pd.DataFrame(compare_rows)
print('=== Start vs End comparison (single lambda=0.5, averaged over seeds) ===')
print()

# Display start rows
start = compare_df[compare_df['phase'] == 'start'].sort_values('method')
end = compare_df[compare_df['phase'] == 'end'].sort_values('method')

print('--- Iteration 0 (all methods should be identical if same init) ---')
show_cols = ['method'] + [c for c in cos_cols + ['loss_dec', 'loss_pred', 'loss_fair'] if c in start.columns]
print(start[show_cols].to_string(index=False, float_format='%.4f'))

print(f'\n--- Final iteration ---')
print(end[show_cols].to_string(index=False, float_format='%.4f'))

In [ ]:
# Cell 13: Save all ablation data to CSV

stage_df_nocont.to_csv(os.path.join(RESULTS_DIR, 'stage_no_continuation.csv'), index=False)
iter_df_nocont.to_csv(os.path.join(RESULTS_DIR, 'iter_no_continuation.csv'), index=False)

stage_df_single.to_csv(os.path.join(RESULTS_DIR, 'stage_single_lambda.csv'), index=False)
iter_df_single.to_csv(os.path.join(RESULTS_DIR, 'iter_single_lambda.csv'), index=False)

for ft, (sdf, idf) in results_by_fairness.items():
    sdf.to_csv(os.path.join(RESULTS_DIR, f'stage_fairness_{ft}.csv'), index=False)
    idf.to_csv(os.path.join(RESULTS_DIR, f'iter_fairness_{ft}.csv'), index=False)

print(f'All results saved to {RESULTS_DIR}/')
!ls -la {RESULTS_DIR}/